# 1603. Design Parking System

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** design, counting, simulation
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-parking-system/)

Design a parking system for a car park with three kinds of space: **big**,
**medium** and **small**, with a fixed number of each.

Implement the `ParkingSystem` class:

- `ParkingSystem(big, medium, small)` initialises the object with the number of
  slots of each size.
- `addCar(carType)` checks whether there is a free slot of type `carType` for the
  car that just drove up. `carType` is `1`, `2` or `3` for big, medium and small.
  **A car can only park in a space of its own size.** If there is no free space,
  return `false`, otherwise park the car in that space and return `true`.

---

### Example

```
Input:  ["ParkingSystem", "addCar", "addCar", "addCar", "addCar"]
        [[1, 1, 0],       [1],      [2],      [3],      [1]]
Output: [null,            true,     true,     false,    false]

ParkingSystem p = new ParkingSystem(1, 1, 0);
p.addCar(1);   // true  - there is 1 big slot, now 0
p.addCar(2);   // true  - there is 1 medium slot, now 0
p.addCar(3);   // false - there are no small slots
p.addCar(1);   // false - the only big slot is taken
```

---

### Constraints

- `0 <= big, medium, small <= 1000`
- `carType` is `1`, `2` or `3`
- At most `1000` calls will be made to `addCar`

The smallest design problem in the repo, and the right one to start a design
track with: there is no algorithm here at all. The entire problem is *what state
do you keep, and how do you index it* - and it still has two traps that have bitten
you before in #622 and #379.

## Before you write anything

**1.** The obvious version is three fields - `self.big`, `self.medium`,
`self.small` - and an `addCar` that is one `if` per type. Write it. Now count:
how many lines change if the car park adds a fourth size next year? Then write
the version where the three counts live in **one** list and `addCar` has no `if`
on the type at all. That difference is the whole problem.

**2.** `carType` is `1`, `2`, `3`. A Python list is indexed `0`, `1`, `2`.
Somewhere a `- 1` has to appear. Where do you put it so it appears **once**, and
what exactly does `addCar(3)` do if you forget it? (Be specific: wrong answer,
crash, or silently parking in the wrong bay?)

**3.** `addCar` must return a **bool**. But this also "works":

```python
return self.slots[carType - 1]      # returns the count left, not True/False
```

LeetCode accepts it, because `0 == False` and `3 == True` in Python. Say why that
is still a bug, and what it costs the first person who writes
`if p.addCar(1) is True:`. (The harness type-checks the return for exactly this
reason - it is the one thing the judge cannot see.)

Then the sharper version of the same question: `0 <= big`, so a car park with
**zero** big slots is legal. What must `addCar(1)` return, and what must it *not*
do to the counter? A version that decrements first and checks afterwards gives the
right answers forever while quietly driving the count to `-1`, `-2`, `-3` - which
nothing notices until you add `leave()` at the bottom of this notebook.

**4.** `addCar` returns a `bool` **and** mutates state - but only sometimes. On
the `False` path, what must **not** happen? Write the method so the mutation and
the `return True` cannot get separated.

**5.** How would you test it? Every call returns a bool, so unlike #707 you can
see the answer. But two returns of `True` look identical whether or not the
counter went down. What sequence of calls makes a missing decrement visible, and
at which call? (Hint: it is the same shape as "fill it exactly to capacity, then
one more".)

## Two routes

**A - one list, indexed by type** *(write this first)*

```
self.slots = [big, medium, small]
```

`addCar` is then three lines with no branch on the type: compute the index once,
compare against `0` **explicitly**, decrement, return. `O(1)` time, `O(1)` space,
and adding a fourth size is a change to `__init__` only.

**B - a dict keyed by the type itself**

```
self.slots = {1: big, 2: medium, 3: small}
```

Now there is no `- 1` anywhere in the class, so question 2's off-by-one cannot
exist. You pay one hash lookup instead of one index, which is nothing, and you buy
the ability to key by something that is not a small integer - `"electric"`,
`"disabled"`, `"motorcycle"` - the moment the car park gets more interesting.

> **The lesson is indexing, not counting.** Route A is what an interviewer expects
> and it is correct. Route B is what you write when the "type" stops being
> `1, 2, 3` and starts being a name - which in a real parking system it always is.
> Write A, then write B, and notice that only one of them survives the feature
> request.

In [ ]:
class ParkingSystem:

    def __init__(self, big: int, medium: int, small: int):
        pass

    def addCar(self, carType: int) -> bool:
        pass

### The test harness

`addCar` returns a bool, so the answers are visible - but the *state* is not, and
a missing decrement produces a correct-looking `True` at the moment you cause it
and a wrong `True` several calls later.

So `check` replays a call sequence against your class **and** against a plain
list of three counters as the model, and compares every return. It also
type-checks the result: `1` and `True` are equal in Python (`1 == True` is
`True`), so a method that returns a count instead of a bool would slip through a
naive comparison. The harness catches it.

`stress` builds long random sequences of calls at small capacities - which is
where fence-post bugs live - and checks them the same way. Run this cell; don't
edit it.

In [ ]:
import random


def check(caps, calls):
    '''Replay addCar(calls) against ParkingSystem(*caps) and a counter model.

    Returns (ok, log).
    '''
    log = []
    try:
        ps = ParkingSystem(*caps)
    except Exception as e:
        return False, [f"   !! ParkingSystem{tuple(caps)} raised {type(e).__name__}: {e}"]

    model = list(caps)                       # [big, medium, small] still free
    log.append(f"ParkingSystem(big={caps[0]}, medium={caps[1]}, small={caps[2]})")

    for t in calls:
        want = model[t - 1] > 0
        if want:
            model[t - 1] -= 1
        try:
            got = ps.addCar(t)
        except Exception as e:
            log.append(f"   !! addCar({t}) raised {type(e).__name__}: {e}")
            return False, log

        name = {1: "big", 2: "medium", 3: "small"}[t]
        log.append(f"addCar({t}) -> {got!r}   ({name}; free now {model})")

        if not isinstance(got, bool):
            log.append(f"   !! addCar({t}) must return a bool, got {type(got).__name__} {got!r}")
            log.append(f"      (careful: 1 == True in Python, so returning a count can look right)")
            return False, log
        if got != want:
            log.append(f"   !! addCar({t}) must return {want!r}, got {got!r}")
            return False, log

    return True, log


def stress(calls, seed=0, cap=2):
    '''Random calls at small capacities - where the fence-post bugs are.'''
    random.seed(seed)
    caps = [random.randint(0, cap) for _ in range(3)]
    return check(caps, [random.randint(1, 3) for _ in range(calls)])


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example",                 (1, 1, 0), [1, 2, 3, 1]),
    ("question 3: every capacity is zero",   (0, 0, 0), [1, 2, 3, 1, 2, 3]),
    ("one big slot, filled then overfilled", (1, 0, 0), [1, 1, 1]),
    ("only mediums exist",                   (0, 3, 0), [2, 2, 2, 2, 1, 3]),
    ("types do not steal from each other",   (1, 1, 1), [3, 3, 2, 2, 1, 1]),
    ("question 2: type 3 is the last index", (0, 0, 2), [3, 3, 3]),
    ("fill big exactly, then one more",      (2, 0, 0), [1, 1, 1]),
    ("a big park, interleaved",              (2, 2, 2), [1, 2, 3, 1, 2, 3, 1, 2, 3]),
]

for name, caps, calls in CASES:
    report(name, *check(caps, calls))

for calls, seed, cap in [(20, 1, 1), (50, 2, 2), (200, 3, 3), (1000, 4, 5)]:
    report(f"stress: {calls} random calls (seed {seed}, capacities 0..{cap})",
           *stress(calls, seed, cap))

# see it, do not just trust the pass/fail
print("\ntrace of the LeetCode example:")
for line in check((1, 1, 0), [1, 2, 3, 1])[1]:
    print("  " + line)

## After it passes

- **Count the lines you would change** to add a fourth vehicle size, in route A and
  in route B. That number is the only honest argument between them.
- **Break it on purpose.** Change `if self.slots[i] == 0` to `if not self.slots[i]`
  and run the tests - which case fails, and why *only* that one? Then change
  `carType - 1` to `carType` and find the call that raises. Making a test suite
  fail deliberately is how you learn what it actually covers.
- **The invariant.** Write it down: *the number of parked cars of each type plus
  the free count equals the original capacity, always.* Then say which line of
  `addCar` could break it. In a real system that invariant is a nightly
  reconciliation job.
- **Make it real.** Add `leave(carType)` - a car drives out. Now the invariant has
  a second way to break: `leave` on a type that was never parked would push the
  free count **above** capacity. Which check does that need, and is it the same
  shape as the `> 0` check in `addCar`?
- Siblings: **#622 Design Circular Queue** (fixed capacity again, and the same
  `0`-is-falsy trap), #379 Design Phone Directory (free/taken slots, but you must
  hand back *which* slot), #1656 Design an Ordered Stream.